# Figure 2: model evidence

$\log p(x \mid K)$ by thermodynamic integration, one bar per path with its estimated error (Monte Carlo and quadrature combined).  Forward paths start from the prior, backward paths from a posterior draw of a finished run; where the two disagree by more than their errors, the temperature ladder was too coarse or too short.  Bars stand on a floor below the lowest path: read differences, not heights.

Input: `RESULTS/ti/*/` (`blat ti`).

**Where to start Jupyter.**  Start it from the top directory (`jupyter lab`), so the file
browser shows the notebooks and the results side by side.  Starting it anywhere else
works too: Jupyter always runs a notebook in the notebook's own folder, and the first
cell finds the top directory from there.

**Which results.**  Set `RESULTS` in the first cell to the results directory of your run,
relative to the top directory: `example/results-quick` (the default) or
`example/results-thorough`.  When `run_final.sh` executes the notebooks it sets it for
you, through the environment variable `BLAT_RESULTS`.

In [ ]:
import os, sys
from pathlib import Path

# the top directory: the nearest folder, from here upwards, that holds the blat package
TOP = next(p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "blat" / "__init__.py").exists())
sys.path.insert(0, str(TOP))

# the results to draw, relative to the top directory (or absolute)
RESULTS = TOP / os.environ.get("BLAT_RESULTS", "example/results-quick")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blat import figures as F

def save(fig, name):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=300, bbox_inches="tight")
    print("wrote", FIGURES / f"{name}.pdf")

In [ ]:
paths = F.load_evidence(RESULTS / "ti")
paths.round(2)

Forward against backward, per $K$:

In [ ]:
gap = paths.pivot_table(index="K", columns="direction", values="log_evidence")
if {"forward", "backward"} <= set(gap.columns):
    gap["backward - forward"] = gap["backward"] - gap["forward"]
gap.round(1)

In [ ]:
fig = F.plot_evidence(paths)
save(fig, "figure2-evidence")